# 03 - Sentence chunking

Sentence chunking follows grammar. It first splits text into complete sentences, then groups those sentences until a word budget is reached.

In RAG, this is useful because a retrieved chunk should be readable evidence. A complete sentence is usually easier to cite than a broken sentence fragment.

In [ ]:
from pathlib import Path
import re

from pypdf import PdfReader

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/sample_documents/AUG242026_05B2203_AAO.pdf").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the Awesome_RAG checkout")
PDF_PATH = ROOT / "data" / "sample_documents" / "AUG242026_05B2203_AAO.pdf"

def normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def load_pdf_pages(path):
    reader = PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = normalize_text(page.extract_text() or "")
        if text:
            pages.append({"page": page_number, "text": text})
    return pages

def chunk_stats(chunks):
    lengths = [len(chunk["text"].split()) for chunk in chunks]
    return {"chunks": len(chunks), "min_words": min(lengths), "avg_words": round(sum(lengths) / len(lengths), 1), "max_words": max(lengths)}

def print_chunks(chunks, limit=5, preview_chars=650):
    for chunk in chunks[:limit]:
        print(f"{chunk['id']} | page {chunk['page_start']}-{chunk['page_end']}")
        print(chunk["text"][:preview_chars])
        print("-" * 100)
    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunks")

pages = load_pdf_pages(PDF_PATH)
{"pages": len(pages), "words": sum(len(page["text"].split()) for page in pages)}

## Sentence chunking function

`split_sentences()` finds sentence boundaries. `sentence_chunks()` then builds chunks by adding one complete sentence at a time.

If adding the next sentence would cross `max_words`, the current chunk is saved first. Then a new chunk starts with the next sentence.

This means sentence chunking does not intentionally cut a sentence in half.

In [ ]:
def split_sentences(text):
    """Split text on simple sentence boundaries for an inspectable teaching example."""
    return [part.strip() for part in re.split(r"(?<=[.!?])\s+", text) if part.strip()]


def sentence_chunks(pages, max_words=180):
    """Group complete sentences into chunks without exceeding max_words when possible."""
    chunks = []

    for page in pages:
        current = []
        current_words = 0

        for sentence in split_sentences(page["text"]):
            count = len(sentence.split())

            if current and current_words + count > max_words:
                chunks.append({
                    "id": f"sentence-{len(chunks) + 1}",
                    "page_start": page["page"],
                    "page_end": page["page"],
                    "text": " ".join(current),
                })
                current = []
                current_words = 0

            current.append(sentence)
            current_words += count

        if current:
            chunks.append({
                "id": f"sentence-{len(chunks) + 1}",
                "page_start": page["page"],
                "page_end": page["page"],
                "text": " ".join(current),
            })

    return chunks

chunks = sentence_chunks(pages, max_words=180)
chunk_stats(chunks)

In [ ]:
print_chunks(chunks, limit=6)

## How to judge it

Sentence chunking is strong when answers are sentence-level facts. It is weaker when the sentence needs the surrounding paragraph or heading to make sense.

Remember: sentence chunking follows grammar, not document layout.